In [2]:
import pandas as pd
from mopac_batch_runner import prepare_batch, run_mopac_batch

In [3]:
df = pd.read_csv('./datasets/12k_pfas_canonical.csv', sep=',')


df_100 = df.head(100)

In [4]:
df_100.head()

,compound_id,SMILES,canonical_SMILES
0,100016-68-0,FC(F)(OC(\F)=C(\F)F)C(F)(F)OC(F)(C(=O)OC)C(F)(F)F,COC(=O)C(F)(OC(F)(F)C(F)(F)OC(F)=C(F)F)C(F)(F)F
1,1000-28-8,FC(C(F)(F)OCC(F)(F)C(F)(F)F)C(F)(F)F,FC(C(F)(F)F)C(F)(F)OCC(F)(F)C(F)(F)F
2,1000339-60-5,ClCc1ccc(cc1F)C(F)(F)F,Fc1cc(C(F)(F)F)ccc1CCl
3,1000509-29-4,FC(F)(C(F)(F)CCN)C(F)(F)F,NCCC(F)(F)C(F)(F)C(F)(F)F
4,1000572-70-2,FC(F)(F)c1ccc(cc1CCO)C(F)(F)F,OCCc1cc(C(F)(F)F)ccc1C(F)(F)F


In [5]:
# Step 1: SMILES → 3D structures → .mop files
prep = prepare_batch(
    df=df_100,
    smiles_col="canonical_SMILES",
    id_col="compound_id",
    keywords="PM7 GEO-OK",
    workdir="mopac_jobs",
    overwrite=True,
    max_workers=6,
)

print(prep[["compound_id", "status", "duration_s"]])

     compound_id status  duration_s
0    100016-68-0  ready       0.110
1      1000-28-8  ready       0.092
2   1000339-60-5  ready       1.945
3   1000509-29-4  ready       0.092
4   1000572-70-2  ready       0.103
..           ...    ...         ...
95  1006486-98-1  ready       0.084
96  1006614-95-4  ready       0.080
97  1006627-74-2  ready       0.077
98   100664-15-1  ready       0.145
99   100700-41-2  ready       0.625

[100 rows x 3 columns]


In [6]:
# Step 2: run MOPAC on all prepared .mop files
results = run_mopac_batch(
    workdir="mopac_jobs",
    overwrite=True,
    timeout=900,
    max_workers=6,
)

print(results[["compound_id", "status", "duration_s"]])

     compound_id   status  duration_s
0      1000-28-8  success       0.645
1    100016-68-0  success       4.572
2   1000339-60-5  success       0.869
3   1000509-29-4  success       0.536
4   1000572-70-2  success       0.681
..           ...      ...         ...
95  1006486-98-1  success       0.990
96  1006614-95-4  success       1.400
97  1006627-74-2  success       1.220
98   100664-15-1  success      14.160
99   100700-41-2  success       4.626

[100 rows x 3 columns]


In [7]:
results[results['arc_file'].isna()]#.to_csv('failed_runs.csv')

,compound_id,smiles,status,mop_file,out_file,arc_file,error_message,return_code,duration_s


In [8]:
results

,compound_id,smiles,status,mop_file,out_file,arc_file,error_message,return_code,duration_s
0,1000-28-8,FC(C(F)(F)F)C(F)(F)OCC(F)(F)C(F)(F)F,success,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,None,0,0.645
1,100016-68-0,COC(=O)C(F)(OC(F)(F)C(F)(F)OC(F)=C(F)F)C(F)(F)F,success,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,None,0,4.572
2,1000339-60-5,Fc1cc(C(F)(F)F)ccc1CCl,success,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,None,0,0.869
3,1000509-29-4,NCCC(F)(F)C(F)(F)C(F)(F)F,success,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,None,0,0.536
4,1000572-70-2,OCCc1cc(C(F)(F)F)ccc1C(F)(F)F,success,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,None,0,0.681
...,...,...,...,...,...,...,...,...,...
95,1006486-98-1,Cc1nn(CC(F)(F)C(F)F)c(C)c1N,success,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,None,0,0.990
96,1006614-95-4,F[C@H]([C@H](F)C(F)(F)C(F)(F)F)C(F)(F)F,success,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,None,0,1.400
97,1006627-74-2,OC/C(I)=C/C(F)(C(F)(F)F)C(F)(F)F,success,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,None,0,1.220
98,100664-15-1,O=P(OCC(F)(F)F)(OCC(F)(F)F)OC(c1ccccc1)C(F)(F)...,success,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,None,0,14.160
